# A first ASMC check

Run this notebook from the `play` directory after building `asmc-tools`.
The example is GNU x86-64 assembly. ASMC assembles, links, and checks it;
you do not need a separate C harness for these register properties.


In [1]:
%%file /tmp/mov42.s
.text
.globl myfun, myfun_end
.type myfun,@function
myfun:
    mov $42, %rax
    ret
.size myfun,.-myfun
myfun_end:
.section .note.GNU-stack,"",@progbits


Overwriting /tmp/mov42.s


`.type` and `.size` give the ELF function boundary. The compiler is `CC`
(default `cc`); `CPPFLAGS`, `CFLAGS`, and `LDFLAGS` can override its options.
The linked file is saved as `target.elf` in the printed output directory.
The generated harness checks the scalar ABI and return/stack safety as well
as the requested postcondition. Registers not fixed by ABI setup are arbitrary.


In [2]:
!../build/bin/asmc /tmp/mov42.s --function myfun --post "out.rax == 42"


Output: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-caimchdl
assemble: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-caimchdl/assemble.log
ELF: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-caimchdl/target.elf
Harness: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-caimchdl/demo.c (x86-64 SysV)
Checking: 1 postcondition(s), safety and scalar ABI obligations.
seed: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-caimchdl/seed.log
lift: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-caimchdl/lift.log
link: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-caimchdl/link.log
cbmc: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-caimchdl/cbmc.log
PASS: all checked properties passed with unwinding assertions


In [3]:
!../build/bin/asmc /tmp/mov42.s --function myfun --post "out.rax == 43"


Output: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-dpcchzti
assemble: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-dpcchzti/assemble.log
ELF: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-dpcchzti/target.elf
Harness: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-dpcchzti/demo.c (x86-64 SysV)
Checking: 1 postcondition(s), safety and scalar ABI obligations.
seed: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-dpcchzti/seed.log
lift: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-dpcchzti/lift.log
link: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-dpcchzti/link.log
cbmc: /home/philip/vibe_coding/asmc/play/asmc-out/myfun-dpcchzti/cbmc.log
  [harness.assertion.1] line 91 post: out.rax == 43
Counterexample (harness/reference values; use --full-trace for machine steps):
Trace for harness.assertion.1:

  demo.c:42  in.rax=0ul
  demo.c:43  in.rcx=0ul
  demo.c:44  in.rdx=0ul
  demo.c:45  in.rbx=0ul
  demo.c:46  in.rsp=4344ul
  demo.c:47  in.rbp=0ul
  demo.c:48  in.

The second check intentionally fails: the counterexample has `out.rax=42`.
`in` and `out` are register snapshots; richer memory contracts need a C harness.
The recent-instruction list now shows actual execution. Temporary declarations
previously inherited assembly locations and made this tiny execution appear twice.
`--full-trace` still exposes the complete CBMC trace.


In [4]:
from pathlib import Path
elf = max(Path("asmc-out").glob("myfun-*/target.elf"), key=lambda p: p.stat().st_mtime)
!objdump -d {elf}
!nm -n {elf}



asmc-out/myfun-dpcchzti/target.elf:     file format elf64-x86-64


Disassembly of section .text:

0000000000401000 <myfun>:
  401000:	48 c7 c0 2a 00 00 00 	mov    $0x2a,%rax
  401007:	c3                   	ret
0000000000401000 T myfun
0000000000401008 T myfun_end
0000000000402000 T __bss_start
0000000000402000 T _edata
0000000000402000 T _end


## When size metadata is missing

Prefer `.type`/`.size` when you can edit the source. Otherwise `--end-address`
is the **exclusive linked virtual address** after the function, not its byte
count or an address in the textual `.s` file. The `myfun_end` label above makes
that address visible in `nm -n` output. Copy its hexadecimal value, including
a `0x` prefix, into a check such as:

```sh
../build/bin/asmc /tmp/mov42.s --function myfun --end-address 0x401008 --post 'out.rax == 42'
```

`0x401008` is only illustrative: use the value in your ELF. `objdump -d` reads
machine code from the linked ELF, so running it on the textual assembly does
not provide these addresses. The explicit end also works on an existing ELF.
